# OpsSim-AI: End-to-End Training Pipeline

This notebook runs the **full OpsSim-AI training pipeline** in a lightweight configuration:

1. **Dataset Generation** — Build SFT + GRPO prompt datasets from cascade scenarios
2. **Supervised Fine-Tuning (SFT)** — Train a LoRA adapter on gold incident-response examples
3. **GRPO Reinforcement Learning** — Refine the policy via environment-grounded reward
4. **Inference** — Load the trained adapter and test on a live scenario
5. **Training Plots** — Visualize reward curves and quality metrics

This uses **minimal steps** to prove the pipeline works end-to-end. For real training, use `submit_hf_job.py` with an L40S/A100 GPU.

**Runtime:** ~15-25 minutes on a free Colab T4 GPU.

## 1. Configuration

Adjust these parameters before running. The defaults are set for a fast demo on a free T4.

In [ ]:
# ====== CONFIGURE THESE ======
BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"
HF_TOKEN = ""  # paste your HF token or leave blank if public model

# SFT settings (lightweight)
SFT_MAX_STEPS = 10
SFT_BATCH_SIZE = 1
SFT_GRAD_ACCUM = 2
SFT_MAX_SEQ_LEN = 512

# GRPO settings (lightweight)
GRPO_MAX_STEPS = 10
GRPO_BATCH_SIZE = 2
GRPO_NUM_GENERATIONS = 2
GRPO_GRAD_ACCUM = 1
GRPO_MAX_PROMPT_LEN = 512
GRPO_MAX_COMPLETION_LEN = 128
GRPO_LEARNING_RATE = 1e-5
GRPO_BETA = 0.01
GRPO_TEMPERATURE = 0.9

# Precision (bf16 on A100/L40S, fp16 on T4)
import torch
PRECISION = "bf16" if torch.cuda.is_bf16_supported() else "fp16"
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"Precision: {PRECISION}")

## 2. Setup Environment

In [ ]:
!pip install -q torch transformers==4.51.3 trl==0.17.0 peft==0.15.2 accelerate==1.6.0
!pip install -q datasets==3.5.0 matplotlib==3.9.2 huggingface_hub
!pip install -q pydantic jmespath

import os
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN

# Clone repo
REPO_DIR = "OpsSim-AI"
if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/nithishgouds/Meta-x-OpenEnv-x-Pytorch-Hackathon.git {REPO_DIR}

os.chdir(REPO_DIR)
print(f"\nWorking directory: {os.getcwd()}")
print(f"Scenarios: {os.path.isfile('tasks/cascade.json')}")

## 3. Generate Training Dataset

This runs `generate_sft_dataset.py` which:
- Loads 10 cascading failure scenarios from `tasks/cascade.json`
- Walks each scenario's optimal solution path
- Generates gold SFT examples + contrast rejection examples + GRPO prompts
- Writes `sft_train.jsonl`, `sft_val.jsonl`, and `grpo_prompts.jsonl`

In [ ]:
!python generate_sft_dataset.py \
    --input tasks/cascade.json \
    --output-dir data/generated \
    --validate

# Show what was generated
import json
summary_path = "data/generated/summary.json"
if os.path.isfile(summary_path):
    with open(summary_path) as f:
        summary = json.load(f)
    print(f"\nDataset summary:")
    print(json.dumps(summary, indent=2))
else:
    # Count lines manually
    for fname in ["sft_train.jsonl", "sft_val.jsonl", "grpo_prompts.jsonl"]:
        fpath = f"data/generated/{fname}"
        if os.path.isfile(fpath):
            with open(fpath) as f:
                count = sum(1 for _ in f)
            print(f"  {fname}: {count} examples")

## 4. Supervised Fine-Tuning (SFT)

Train a LoRA adapter on the gold incident-response examples. This teaches the model:
- Output valid JSON with the 6 required keys
- Pick reasonable actions from the playbook
- Route to the correct domain agent

We use very few steps here just to prove the pipeline runs. Real training uses ~1 epoch.

In [ ]:
SFT_OUTPUT = "outputs/sft-demo"

!python train_sft.py \
    --model {BASE_MODEL} \
    --train-file data/generated/sft_train.jsonl \
    --val-file data/generated/sft_val.jsonl \
    --output-dir {SFT_OUTPUT} \
    --max-steps {SFT_MAX_STEPS} \
    --batch-size {SFT_BATCH_SIZE} \
    --grad-accum {SFT_GRAD_ACCUM} \
    --max-seq-length {SFT_MAX_SEQ_LEN} \
    --logging-steps 1 \
    --save-steps {SFT_MAX_STEPS} \
    --precision {PRECISION}

print(f"\nSFT adapter saved to: {SFT_OUTPUT}")
print(f"Contents: {os.listdir(SFT_OUTPUT)}")

## 5. GRPO Reinforcement Learning

Refine the SFT adapter using Group Relative Policy Optimization with **environment-grounded reward**.

Unlike SFT, the model now:
- Generates multiple completions per prompt
- Gets scored by the actual DevOpsEnv (action quality, sequencing, coordination, etc.)
- Learns from relative advantages within each group

This is the core of the training pipeline — the model improves by interacting with the environment.

In [ ]:
GRPO_OUTPUT = "outputs/grpo-demo"

!python train_grpo.py \
    --model {BASE_MODEL} \
    --sft-adapter {SFT_OUTPUT} \
    --input tasks/cascade.json \
    --prompt-file data/generated/grpo_prompts.jsonl \
    --output-dir {GRPO_OUTPUT} \
    --max-steps {GRPO_MAX_STEPS} \
    --batch-size {GRPO_BATCH_SIZE} \
    --num-generations {GRPO_NUM_GENERATIONS} \
    --grad-accum {GRPO_GRAD_ACCUM} \
    --max-prompt-length {GRPO_MAX_PROMPT_LEN} \
    --max-completion-length {GRPO_MAX_COMPLETION_LEN} \
    --learning-rate {GRPO_LEARNING_RATE} \
    --beta {GRPO_BETA} \
    --temperature {GRPO_TEMPERATURE} \
    --logging-steps 1 \
    --save-steps {GRPO_MAX_STEPS} \
    --precision {PRECISION}

print(f"\nGRPO adapter saved to: {GRPO_OUTPUT}")
print(f"Contents: {os.listdir(GRPO_OUTPUT)}")

## 6. Training Plots

Display training curves generated during GRPO. These are auto-saved to the `plot_data/` subdirectory.

In [ ]:
import glob
from IPython.display import Image, display

# Generate plots from training logs
!python plot_training_logs.py --grpo-dir {GRPO_OUTPUT} --output-dir plots/demo 2>/dev/null || true

# Find all generated plot PNGs (check both the output plot dir and GRPO output)
plot_dirs = ["plots/demo", f"{GRPO_OUTPUT}/plot_data", f"{GRPO_OUTPUT}"]
plot_files = []
for d in plot_dirs:
    plot_files.extend(glob.glob(os.path.join(d, "*.png")))

# Prioritize key plots
priority_names = ["reward_smoothed", "reward_only", "loss", "kl", "quality", "reward_vs_kl"]

def plot_priority(path):
    name = os.path.basename(path).lower()
    for i, key in enumerate(priority_names):
        if key in name:
            return i
    return 100

plot_files = sorted(set(plot_files), key=plot_priority)

if plot_files:
    print(f"Found {len(plot_files)} plots. Showing top results:\n")
    for pf in plot_files[:6]:
        print(f"--- {os.path.basename(pf)} ---")
        display(Image(filename=pf, width=700))
        print()
else:
    print("No plot PNGs found. Checking for raw metrics...")
    metrics_path = os.path.join(GRPO_OUTPUT, "plot_data", "train_metrics.jsonl")
    if os.path.isfile(metrics_path):
        import matplotlib.pyplot as plt
        metrics = []
        with open(metrics_path) as f:
            for line in f:
                metrics.append(json.loads(line))
        if metrics:
            steps = [m.get("step", i) for i, m in enumerate(metrics)]
            losses = [m.get("loss", 0) for m in metrics]
            fig, ax = plt.subplots(1, 1, figsize=(10, 4))
            ax.plot(steps, losses, alpha=0.7)
            ax.set_xlabel("Step")
            ax.set_ylabel("Loss")
            ax.set_title("GRPO Training Loss")
            ax.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.show()
            print(f"Plotted {len(metrics)} logged steps.")
    else:
        print(f"No metrics found at {metrics_path}")

## 7. Quick Inference Test

Load the trained GRPO adapter and run inference on a sample scenario to verify the model produces valid structured output.

In [ ]:
# Run inference using the repo's existing script
!python run_trained_inference.py \
    --base-model {BASE_MODEL} \
    --adapter {GRPO_OUTPUT} \
    --input tasks/cascade.json \
    --scenario-id cascade_001_checkout_meltdown \
    --step-idx 1 \
    --precision {PRECISION} \
    --show-prompt

In [ ]:
# Run a second example at a later step to show multi-step reasoning
!python run_trained_inference.py \
    --base-model {BASE_MODEL} \
    --adapter {GRPO_OUTPUT} \
    --input tasks/cascade.json \
    --scenario-id cascade_001_checkout_meltdown \
    --step-idx 3 \
    --precision {PRECISION}

## 8. Inspect Final Metrics

Check the final quality metrics logged during GRPO training.

In [ ]:
# Check for final metrics from GRPO
final_metrics_path = os.path.join(GRPO_OUTPUT, "final_metrics.json")
if os.path.isfile(final_metrics_path):
    with open(final_metrics_path) as f:
        final_metrics = json.load(f)
    print("GRPO Final Metrics:")
    print(json.dumps(final_metrics, indent=2))
else:
    # Try plot_data summary
    summary_path = os.path.join(GRPO_OUTPUT, "plot_data", "summary.json")
    if os.path.isfile(summary_path):
        with open(summary_path) as f:
            summary = json.load(f)
        print("GRPO Training Summary:")
        print(json.dumps(summary, indent=2))
    else:
        print("No final metrics file found. Check GRPO output for logs.")
        for fname in sorted(os.listdir(GRPO_OUTPUT)):
            fpath = os.path.join(GRPO_OUTPUT, fname)
            size = os.path.getsize(fpath) if os.path.isfile(fpath) else "dir"
            print(f"  {fname}  ({size})")

## 9. Summary

This notebook demonstrated the complete OpsSim-AI pipeline:

| Stage | Script | What it does |
|-------|--------|--------------|
| Data | `generate_sft_dataset.py` | Builds gold + contrast training data from 10 cascade scenarios |
| SFT | `train_sft.py` | LoRA fine-tuning on gold incident-response examples |
| GRPO | `train_grpo.py` | RL refinement using env-grounded reward (13-component) |
| Inference | `run_trained_inference.py` | Tests the trained adapter on live scenarios |
| Plots | `plot_training_logs.py` | Generates reward/loss/quality training curves |

For production training, use `submit_hf_job.py` to launch on HuggingFace Jobs with an A100 GPU and 500+ GRPO steps.

In [ ]:
# Cleanup GPU memory
import torch
torch.cuda.empty_cache()
print("Done! GPU memory freed.")